# Kandinsky V2A kalite karşılaştırması — Pro Distill

Bu notebook, çalışan Lite Distill denemesine dokunmadan Pro Distill çıktısı üretir. A/B dinlemesinde Lite denemendeki **aynı video, prompt, süre ve seed** değerlerini kullan.

Bu deney ses kalitesini ölçer: doğallık, bozulma/gürültü, olayın ses karakteri ve istenmeyen sesler. Senkronizasyonu yalnızca kontrol ölçütü olarak puanla. Pro checkpoint daha büyük olsa da daha iyi sonuç garanti değildir.

Pro Distill resmi model kartında 30B parametreli, 10 adımlı checkpoint olarak listelenir. Resmî A100 performans notları A100 80 GB kullanır. Kaynaklar:
- https://huggingface.co/kandinskylab/Kandinsky-6.0-Pro-distill-5s-Diffusers
- https://github.com/kandinskylab/kandinsky-6/blob/main/docs/performance/a100.md
- https://github.com/kandinskylab/kandinsky-6/blob/main/docs/generation_modes.md

## 1 — Colab GPU ve disk alanı

Colab Runtime ayarlarından A100 80 GB seç. Pro ağırlıkları büyüktür; indirmeden önce boş alanı kontrol ediyoruz. En az 90 GiB boş alan yoksa devam etme. API anahtarı gerekmez.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile
from pathlib import Path

print("Python:", sys.version)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("Colab Runtime > Change runtime type > GPU seçip yeniden çalıştır.")
if sys.version_info < (3, 13):
    raise RuntimeError("Resmî Kandinsky pipeline Python 3.13 veya 3.14 istiyor.")
gpu_line = gpu.stdout.strip().splitlines()[0]
if "A100" not in gpu_line or "80" not in gpu_line:
    raise RuntimeError("Bu deney A100 80 GB için hazırlanmıştır. Colab GPU seçimini kontrol et.")
free_gib = shutil.disk_usage("/content").free / (1024**3)
print(f"/content boş alanı: {free_gib:.1f} GiB")
if free_gib < 90:
    raise RuntimeError("Pro ağırlıkları ve çalışma dosyaları için en az 90 GiB boş alan bırak.")

## 2 — Resmî ortamı kur ve Pro Distill'i indir

Bu kurulum ayrı bir Colab runtime'ında çalıştırılabilir. SDPA kullanıyoruz; SageAttention kurulumuna bağlı değiliz. Büyük ağırlık indirmesi zaman alabilir.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kandinskylab/kandinsky-6.git
%cd /content/kandinsky-6
!python -m pip -q install uv
!curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin
!just setup
print("Kaynak commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
!just download pro-distill
print("Pro Distill indirildi.")

## 3 — Lite denemesiyle aynı girdileri kullan

EVENT_PROMPT, SEED ve GENERATION_SECONDS alanlarını Lite çıktısında kullandığın değerlerle eşitle. İlk üretim prompt'un farklıysa aşağıdaki varsayılanı aynen değiştir. Prompt genişletme kapalı kalır.

In [ ]:
from google.colab import files
from IPython.display import display, Video, Audio

EVENT_PROMPT = "footsteps"  # Lite denemendeki prompt'u buraya aynen yaz.
SEED = 42
GENERATION_SECONDS = 5

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Bir MP4 video yüklemedin.")
filename = next(iter(uploaded))
source_video = Path("/content") / Path(filename).name
source_video.write_bytes(uploaded[filename])
probe = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(source_video)], capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError("Video okunamadı: " + probe.stderr[-1200:])
source_duration = float(probe.stdout.strip())
print(f"Video: {filename} | {source_duration:.2f}s | prompt: {EVENT_PROMPT} | seed: {SEED}")
if source_duration < GENERATION_SECONDS:
    raise ValueError("Kaynak video üretim süresinden kısa.")
input_video = Path("/content/kandinsky_v2a_pro_input.mp4")
prep = subprocess.run(["ffmpeg", "-y", "-i", str(source_video), "-t", str(GENERATION_SECONDS), "-map", "0:v:0", "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "24", str(input_video)], capture_output=True, text=True)
if prep.returncode != 0:
    raise RuntimeError("Üretim videosu hazırlanamadı: " + prep.stderr[-1500:])
input_sha256 = hashlib.sha256(input_video.read_bytes()).hexdigest()
print("Girdi video SHA-256:", input_sha256)

## 4 — Pro Distill ile üret

Lite ile aynı seed, prompt ve süreyi kullanıyoruz. Distill checkpoint'in resmî ayarı 10 adımdır; bu deneyde adım sayısını ayrıca değiştirmiyoruz. Pro'nun Lite'tan iyi çıkacağını varsaymıyoruz.

In [ ]:
DEVICE_CONFIG = Path("/content/kandinsky-6/kandinsky/configs/devices/a100-pro-distill-sdpa.yaml")
DEVICE_CONFIG.write_text(
    "checkpoint: pro-distill\ncache:\n  mode: none\noffload:\n  strategy: module\ncompile:\n  strategy: none\nattention:\n  engine: sdpa\ntext_embedder:\n  quantized_qwen: false\nbeautifier:\n  name: none\n",
    encoding="utf-8",
)
OUTPUT_MP4 = Path("/content/kandinsky_v2a_pro_distill_output.mp4")
cmd = ["just", "generate", EVENT_PROMPT, "--config", str(DEVICE_CONFIG), "--attention-engine", "sdpa", "--video-file", str(input_video), "--time-length", str(GENERATION_SECONDS), "--out", str(OUTPUT_MP4), "--seed", str(SEED)]
print("Üretim başlıyor:", " ".join(cmd))
started = time.time()
result = subprocess.run(cmd, cwd="/content/kandinsky-6", text=True, capture_output=True)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-6000:])
    raise RuntimeError("Pro Distill üretimi başarısız oldu; hata metnini incele.")
if not OUTPUT_MP4.is_file():
    raise FileNotFoundError(f"Çıktı bulunamadı: {OUTPUT_MP4}")
elapsed_seconds = time.time() - started
print(f"Üretim süresi: {elapsed_seconds:.1f} saniye")

## 5 — Dinle ve puanla

Videoyu ve WAV'i dinle; sonra Lite çıktısıyla yan yana karşılaştır. WAV tek sürekli akıştır. Kandinsky çıktısı olayları ayrı one-shot dosyaları olarak teslim etmez.

In [ ]:
display(Video(data=OUTPUT_MP4.read_bytes(), embed=True, mimetype="video/mp4"))
OUTPUT_WAV = Path("/content/kandinsky_v2a_pro_distill_audio.wav")
extract = subprocess.run(["ffmpeg", "-y", "-i", str(OUTPUT_MP4), "-vn", "-ac", "2", "-ar", "44100", str(OUTPUT_WAV)], capture_output=True, text=True)
if extract.returncode != 0:
    raise RuntimeError("WAV çıkarılamadı: " + extract.stderr[-1500:])
display(Audio(filename=str(OUTPUT_WAV)))

## 6 — Değerlendirme raporu

Puanlar kişisel dinleme notlarıdır; tek örnek genel kaliteyi kanıtlamaz. Senkronizasyon kontrol puanıdır. “Tek sürekli dosya yeterli mi?” ürünün editlenebilir one-shot ihtiyacını ayrıca kaydeder.

In [ ]:
scores = {
    "audio_quality_1_to_5": None,
    "naturalness_1_to_5": None,
    "artifacts_or_noise_1_to_5": None,  # 1=çok az, 5=çok belirgin
    "event_match_1_to_5": None,
    "sync_1_to_5_control_metric": None,
    "unwanted_music_or_speech": None,
    "single_continuous_audio_asset_sufficient": None,
    "need_independent_one_shots": None,
    "listening_notes": "",
}
report = {
    "experiment": "kandinsky_v2a_quality_checkpoint_ab_v1",
    "checkpoint": "kandinskylab/Kandinsky-6.0-Pro-distill-5s-Diffusers",
    "source_repo": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": subprocess.check_output(["git", "-C", "/content/kandinsky-6", "rev-parse", "HEAD"], text=True).strip(),
    "input_video_name": filename,
    "input_video_sha256": input_sha256,
    "event_prompt": EVENT_PROMPT,
    "prompt_expansion": "disabled",
    "seed": SEED,
    "duration_seconds": GENERATION_SECONDS,
    "gpu": gpu.stdout.strip(),
    "generation_seconds": round(elapsed_seconds, 2),
    "audio_output": "one continuous WAV track extracted from generated MP4; independent semantic one-shots are not supplied",
    "scores": scores,
    "commercial_clearance": "pending_full_dependency_and_data_provenance_review",
}
REPORT = Path("/content/kandinsky_v2a_pro_distill_report.json")
REPORT.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
BUNDLE = Path("/content/kandinsky_v2a_pro_distill_bundle.zip")
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT_MP4, arcname=OUTPUT_MP4.name)
    archive.write(OUTPUT_WAV, arcname=OUTPUT_WAV.name)
    archive.write(REPORT, arcname=REPORT.name)
print(json.dumps(report, ensure_ascii=False, indent=2))
files.download(str(BUNDLE))